# SeaEngine Balancing Metrics (Saved Model Self-Play)

This notebook assumes training is already done (e.g. `~/balance_inputs/model_ep_10000.pt`).
It loads that model from `~/balance_inputs/` and runs about 2000 evaluation matches in RL-vs-RL self-play to produce balancing metrics.

In [ ]:
import importlib
import os
import sys
import time
from pathlib import Path
from shutil import which

run_start_ts = time.time()
home = Path.home()
if str(home) not in sys.path:
    sys.path.insert(0, str(home))

dotnet_cmd = which("dotnet")
if dotnet_cmd is None:
    fallback = home / ".dotnet" / ("dotnet.exe" if os.name == "nt" else "dotnet")
    if fallback.exists():
        dotnet_cmd = str(fallback)
if dotnet_cmd:
    os.environ["DOTNET_CMD"] = dotnet_cmd
    dotnet_root = str(Path(dotnet_cmd).resolve().parent)
    os.environ.setdefault("DOTNET_ROOT", dotnet_root)
    os.environ.setdefault("DOTNET_ROOT_X64", dotnet_root)

for module_name in list(sys.modules):
    if module_name == "RL_AI" or module_name.startswith("RL_AI."):
        del sys.modules[module_name]
importlib.invalidate_caches()

from RL_AI.SeaEngine.experiment import run_saved_model_balance_experiment
from RL_AI.SeaEngine import experiment as seaengine_experiment_module
print(f"experiment source: {seaengine_experiment_module.__file__}")

In [ ]:
# If MODEL_PATH does not exist, fallback to the latest *.pt under ~/balance_inputs
MODEL_PATH = str(Path.home() / "balance_inputs" / "model_ep_10000.pt")

model_path_obj = Path(MODEL_PATH)
if not model_path_obj.exists():
    input_dir = Path.home() / "balance_inputs"
    candidates = sorted(input_dir.glob("*.pt"), key=lambda p: p.stat().st_mtime)
    if not candidates:
        candidates = sorted((Path.home() / "RL_AI" / "models").glob("model_ep_*.pt"), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError("No model file found in ~/balance_inputs or ~/RL_AI/models")
    model_path_obj = candidates[-1]

print(f"using model: {model_path_obj}")

In [ ]:
from datetime import datetime
import zipfile

TOTAL_MATCHES = 2000
MAX_TURNS = 100
SEED = 7

balance = run_saved_model_balance_experiment(
    model_path=str(model_path_obj),
    total_matches=TOTAL_MATCHES,
    max_turns=MAX_TURNS,
    seed=SEED,
    device="auto",
    opponent_mode="self",
    include_history=False,
)

print("=== Saved Model Balance Result ===")
print(f"summary report: {balance['summary_report_path']}")
print(balance["aggregate"])

# Zip newly created txt logs from this run
log_dir = Path.home() / "RL_AI" / "log"
new_logs = sorted([p for p in log_dir.glob("*.txt") if p.stat().st_mtime >= run_start_ts - 1.0], key=lambda p: p.name)
if new_logs:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_path = log_dir / f"log_{ts}.zip"
    with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in new_logs:
            zf.write(p, arcname=p.name)
    print(f"log zip: {zip_path}")
else:
    print("no new txt logs to zip")